In [1]:
%matplotlib inline
from __future__ import division
import numpy as np
from numpy.random import rand
import matplotlib.pyplot as plt
from numba import jit
import os
import re
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
#----------------------------------------------------------------------
##  BLOCK OF FUNCTIONS USED IN THE MAIN CODE
#----------------------------------------------------------------------
#@jit(nopython=True)
def initialstate(N, type):   
    ''' generates a random spin configuration for initial condition'''
    
    if type == 'random':
        state = 2*np.random.randint(2, size=(N,N))-1
        
    if type == 'half up':
        state = np.zeros((N,N))+1
        state[int(N/2):, :]=-1
    
    if type == 'checkerboard':
        state = np.ones((N, N))
        # fill with 1 the alternate cells in rows and columns
        state[1::2, ::2] = -1
        state[::2, 1::2] = -1
        
    return state

@jit(nopython=True)
def mcmove(config, beta, J, N):
    '''Monte Carlo move using Metropolis algorithm '''
    for i in range(N):
        for j in range(N):
                a = np.random.randint(0, N)
                b = np.random.randint(0, N)
                c = np.random.randint(0, N)
                d = np.random.randint(0, N)
                
                s1 =  config[a, b]
                s2 =  config[c, d]
                
                nb1 = config[(a+1)%N,b] + config[a,(b+1)%N] + config[(a-1)%N,b] + config[a,(b-1)%N]
                nb2 = config[(c+1)%N,d] + config[c,(d+1)%N] + config[(c-1)%N,d] + config[c,(d-1)%N]
                
                final = -J*s2*nb1 + -J*s1*nb2 
                initial = -J*s1*nb1 + -J*s2*nb2
                
                cost = final - initial
                if cost < 0:
                    s1 = config[c,d] 
                    s2 = config[a,b]
                elif rand() < np.exp(-cost*beta):
                    s1 = config[c,d] 
                    s2 = config[a,b]
                config[a, b] = s1
                config[c, d] = s2
    return config

@jit(nopython=True)
def calcEnergy(config,  J, N):
    '''Energy of a given configuration'''
    energy = 0
    for i in range(len(config)):
        for j in range(len(config)):
            S = config[i,j]
            nb = config[(i+1)%N, j] + config[i,(j+1)%N] + config[(i-1)%N, j] + config[i,(j-1)%N]
            energy += -J*nb*S
    return energy/4.


@jit(nopython=True)
def calcMag(config):
    '''Magnetization of a given configuration'''
    mag = np.sum(config)
    return mag

def makeFigSaveData( config, N, beta, folder):
    
    X, Y = np.meshgrid(range(N), range(N))
    plt.figure() 
    #plt.setp(sp.get_yticklabels(), visible=False)
    #plt.setp(sp.get_xticklabels(), visible=False)      
    plt.pcolormesh(X, Y, config, cmap=plt.cm.binary);
    # getting current axes
    a = plt.gca()

    # set visibility of x-axis as False
    xax = a.axes.get_xaxis()
    xax = xax.set_visible(False)

    # set visibility of y-axis as False
    yax = a.axes.get_yaxis()
    yax = yax.set_visible(False)
    plt.axis('tight') 
    
    fname = 'beta_'+str(beta)+'_'
    
    if not os.path.isdir('figures/'+folder): 
        # if the demo_folder2 directory is  
        # not present then create it. 
        os.makedirs('figures/'+folder) 
        
    plt.savefig('./figures/'+folder+'/'+str(fname)+'.png')
    
    np.savetxt('./'+folder+'/'+str(fname)+'.txt', config)
    #plt.show()
    plt.close()


    

In [ ]:
## change these parameters for a smaller (faster) simulation 
nt      = 4000       #  number of temperature points
N_vals  = [40]         #  size of the lattice, N x N
eqSteps = 10000       #  number of MC sweeps for equilibration
mcSteps = 50000       #  number of MC sweeps for calculation

type_folder = 'antiferro_COP'
J= -1


T       = np.linspace(1.0, 3.5, nt);
#T=[1, 2, 2.269, 3, 4]
E,M,C,X = np.zeros(nt), np.zeros(nt), np.zeros(nt), np.zeros(nt)
#n1, n2  = 1.0/(mcSteps*N*N), 1.0/(mcSteps*mcSteps*N*N) 
# divide by number of samples, and by system size to get intensive values

In [4]:
#----------------------------------------------------------------------
#  MAIN PART OF THE CODE
#----------------------------------------------------------------------
for N in N_vals:
    folder =type_folder+'/data_N_'+str(N)+'_train'
    # checking if the directory exist or not. 
    if not os.path.isdir(folder): 
        os.makedirs(folder) 
    
    for tt in range(nt):
        #E1 = M1 = E2 = M2 = 0
        beta=1.0/T[tt];
        fname = 'beta_'+str(beta)+'_'
        
        if os.path.isfile('./'+folder+'/'+str(fname)+'.txt'):
            config = np.loadtxt('./'+folder+'/'+str(fname)+'.txt') #load data if file exists
            #print('true')
        else: config = initialstate(N, 'half up') #or make new configuration
        #print(config)
        
        for i in range(eqSteps):         # equilibrate
            config = mcmove(config, beta, J, N)           # Monte Carlo moves

        for i in range(mcSteps):
            config = mcmove(config, beta, J, N)           


        makeFigSaveData(config, N, beta, folder)
